In [ ]:
# In[1]:

import pandas as pd
import pytz

# load metric data
metric_df = pd.read_csv('metric.csv')

# convert timestamp to timezone-aware datetime in UTC then to Asia/Shanghai (UTC+8)
metric_df['timestamp_dt'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True).dt.tz_convert(pytz.timezone('Asia/Shanghai'))

# ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# compute global thresholds/stats per cmdb_id and kpi_name using full available data
grouped_stats_full = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(
        count='count',
        mean='mean',
        min='min',
        max='max',
        p50=lambda x: x.quantile(0.50),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    )
    .reset_index()
)

# Round numeric columns for compactness
num_cols = ['mean', 'min', 'max', 'p50', 'p90', 'p95', 'p99']
grouped_stats_full[num_cols] = grouped_stats_full[num_cols].round(6)

# sort by count descending
grouped_stats_full = grouped_stats_full.sort_values('count', ascending=False).reset_index(drop=True)

# prepare compact aggregated table limited to top 50 rows for display
agg_top50 = grouped_stats_full.head(50)

# list of unique components and number of distinct KPIs per component
component_kpi_counts = (
    metric_df
    .groupby('cmdb_id')['kpi_name']
    .nunique()
    .reset_index(name='kpi_count')
    .sort_values('kpi_count', ascending=False)
    .reset_index(drop=True)
)

# Display compact outputs: aggregated top50 and component KPI counts
agg_top50, component_kpi_counts

```
Out[1]:
```
```python
# Prepare a concise plain-English summary based on the previously computed results
n_groups = grouped_stats_full.shape[0]
n_components = component_kpi_counts.shape[0]
top_components = component_kpi_counts.head(5).to_dict(orient='records')

summary = (
    f"Summary of metric.csv global thresholds:\n"
    f"- Total component-KPI series analyzed: {n_groups}.\n"
    f"- Number of unique components: {n_components}.\n"
    f"- Top components by number of distinct KPIs (component: kpi_count): "
    + ", ".join([f\"{r['cmdb_id']}:{r['kpi_count']}\" for r in top_components])
    + ".\n"
    f"- For each component-KPI series we computed: count, mean, min, max, p50 (median), p90, p95, and p99 using the full available data.\n"
    f"- The full grouped table is stored in variable 'grouped_stats_full'. A compact aggregated table (top 50 rows by count) is in 'agg_top50'.\n"
    f"- To inspect more details, call grouped_stats_full.head(X) or component_kpi_counts.head(X) in the kernel."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count        mean        min       max         p50         p90         p95         p99
0   apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31    0.000000    0.00000    0.0000    0.000000    0.000000    0.000000    0.000000
1   apache02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31    0.000000    0.00000    0.0000    0.000000    0.000000    0.000000    0.000000
2   apache02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31    0.000000    0.00000    0.0000    0.000000    0.000000    0.000000    0.000000
3   apache02                            OSLinux-CPU_CPU_CPULoad     31    0.041290    0.00000    0.1700    0.020000    0.130000    0.145000    0.164000
4   apache02                         OSLinux-CPU_CPU_CPUSysTime     31    0.359951    0.22429    0.4858    0.374049    0.440455    0.465502    0.479711
..       ...                                                ...    ...         ...        ...       ...         ...         ...         ...         ...
45  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31    0.000000    0.00000    0.0000    0.000000    0.000000    0.000000    0.000000
46  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31  143.000000  143.00000  143.0000  143.000000  143.000000  143.000000  143.000000
47  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31    4.129032    0.00000   11.0000    4.000000    7.000000    9.500000   10.700000
48  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31  500.000000  500.00000  500.0000  500.000000  500.000000  500.000000  500.000000
49  Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter     31    0.000000    0.00000    0.0000    0.000000    0.000000    0.000000    0.000000

[50 rows x 10 columns],          cmdb_id  kpi_count
0        Mysql01        224
1        Mysql02        224
2        Redis01        115
3        Redis02        110
4       Tomcat03        110
..           ...        ...
24  ServiceTest9          4
25  ServiceTest5          4
26  ServiceTest4          4
27  ServiceTest6          4
28  ServiceTest7          4

[29 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

# reuse existing variables: metric_df, grouped_stats_full (from previous steps in the kernel)

# 1) Define incident window in Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
window_start = pd.Timestamp('2021-03-09 11:00:00', tz=tz)
window_end   = pd.Timestamp('2021-03-09 11:30:00', tz=tz)

# 2) Compute global p5 per (cmdb_id, kpi_name) using full metric_df (thresholds must be global BEFORE filtering)
p5_df = metric_df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).reset_index(name='p5')

# Merge p5 into the stored grouped_stats_full (which already contains p95 from earlier)
# Make a copy to avoid mutating original unexpectedly
grouped_stats_full_with_p5 = grouped_stats_full.merge(p5_df, on=['cmdb_id', 'kpi_name'], how='left')

# 1 continued) Filter metric_df to the incident window using timestamp_dt (already timezone-aware)
filtered_window = metric_df[
    (metric_df['timestamp_dt'] >= window_start) &
    (metric_df['timestamp_dt'] <= window_end)
].copy()

# 2 continued) Join p95 and p5 to each record in the filtered window
filtered_with_thresh = filtered_window.merge(
    grouped_stats_full_with_p5[['cmdb_id', 'kpi_name', 'p5', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# 3) Mark anomaly: value >= p95 OR value <= p5
# Only mark True when p5 and p95 are present; otherwise False
filtered_with_thresh['anomaly_flag'] = (
    (~filtered_with_thresh['value'].isna()) &
    (~filtered_with_thresh['p95'].isna()) &
    (~filtered_with_thresh['p5'].isna()) &
    ((filtered_with_thresh['value'] >= filtered_with_thresh['p95']) |
     (filtered_with_thresh['value'] <= filtered_with_thresh['p5']))
)

# 5) Store full anomaly records table (with requested columns) in kernel variable 'anomalies_window'
anomalies_window = filtered_with_thresh.loc[:, ['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p5', 'p95', 'anomaly_flag', 'timestamp_dt']].copy()

# Keep timestamp_dt for earliest-time aggregation but the stored table includes required columns too
# (timestamp_dt kept for aggregation convenience; it's fine to keep in memory)

# 4) Produce compact aggregate of anomalies per cmdb_id:
# Filter records that are anomalies within the window
anom_only = anomalies_window[anomalies_window['anomaly_flag']].copy()

# Total anomaly count and earliest anomaly timestamp per component
comp_counts = (
    anom_only
    .groupby('cmdb_id')
    .agg(
        total_anomalies=('anomaly_flag', 'sum'),
        earliest_anom_ts=('timestamp_dt', 'min')
    )
    .reset_index()
)

# Top 5 kpi_name with their anomaly counts for each component
kpi_counts = (
    anom_only
    .groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='kpi_anom_count')
)

# For each component, build a compact string of top 5 KPIs: "kpi1:count; kpi2:count; ..."
def top_kpis_string(df, topn=5):
    df_sorted = df.sort_values('kpi_anom_count', ascending=False).head(topn)
    return "; ".join([f"{r['kpi_name']}:{int(r['kpi_anom_count'])}" for _, r in df_sorted.iterrows()])

top_kpis_per_comp = (
    kpi_counts
    .groupby('cmdb_id')
    .apply(lambda d: pd.Series({'top_kpis': top_kpis_string(d, topn=5)}))
    .reset_index()
)

# Join to comp_counts
anomalies_agg = comp_counts.merge(top_kpis_per_comp, on='cmdb_id', how='left')

# Format earliest timestamp as readable string in Asia/Shanghai
anomalies_agg['earliest_anom_ts'] = anomalies_agg['earliest_anom_ts'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Select and sort by total_anomalies desc, limit to top 20
anomalies_agg_top20 = anomalies_agg[['cmdb_id', 'total_anomalies', 'earliest_anom_ts', 'top_kpis']].sort_values('total_anomalies', ascending=False).head(20).reset_index(drop=True)

# 5 continued) Total number of anomaly records detected in the window
total_anomalies = int(anom_only.shape[0])

# Clean up anomalies_window for storage per instruction: keep exact requested columns (timestamp, cmdb_id, kpi_name, value, p5, p95, anomaly_flag)
anomalies_window = anomalies_window[['timestamp', 'cmdb_id', 'kpi_name', 'value', 'p5', 'p95', 'anomaly_flag', 'timestamp_dt']].copy()

# Final compact outputs: aggregated top20, total anomalies, and shape of full anomalies table (stored in anomalies_window)
anomalies_agg_top20, total_anomalies, anomalies_window.shape

```
Out[2]:
```
# Plain-English summary of findings (based on the computed aggregates in the kernel)
summary = (
    "Summary of anomalies in the incident window (2021-03-09 11:00:00 to 2021-03-09 11:30:00 UTC+8):\n"
    "- Total anomaly records detected in the window: 18,704.\n"
    "- Top components by anomaly count (component: count):\n"
    "  1) Mysql01: 2751\n"
    "  2) Mysql02: 2666\n"
    "  3) Redis01: 1558\n"
    "  4) Redis02: 1556\n"
    "  5) Tomcat02: 1237\n"
    "  6) Tomcat04: 1173\n"
    "  7) Tomcat01: 1124\n"
    "  8) Tomcat03: 1120\n"
    "  9) IG02: 854\n"
    " 10) IG01: 837\n"
    "- For most top components the earliest anomaly in the window occurred at 2021-03-09 11:00:00 (UTC+8).\n"
    "- For each component in the aggregate we also recorded the top 5 KPIs that contributed most anomalies (shown in 'anomalies_agg_top20').\n"
    "- Stored variables for further inspection in the kernel:\n"
    "  * anomalies_window  -- full anomaly records table for the window (columns: timestamp, cmdb_id, kpi_name, value, p5, p95, anomaly_flag, timestamp_dt).\n"
    "  * anomalies_agg_top20 -- compact per-component aggregate (top 20 components by anomaly count).\n"
    "  * grouped_stats_full -- global per-(cmdb_id,kpi_name) statistics (used to compute p5/p95 thresholds).\n"
    "Use these variables to inspect specific records or KPIs as needed."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id  total_anomalies         earliest_anom_ts                                           top_kpis
0         Mysql01             2751  2021-03-09 11:00:00 CST  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
1         Mysql02             2666  2021-03-09 11:00:00 CST  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
2         Redis01             1558  2021-03-09 11:00:00 CST  OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies:...
3         Redis02             1556  2021-03-09 11:00:00 CST  OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount:31;...
4        Tomcat02             1237  2021-03-09 11:00:00 CST  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...
5        Tomcat04             1173  2021-03-09 11:00:00 CST  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...
6        Tomcat01             1124  2021-03-09 11:00:00 CST  Tomcat-Sessions_7441--_SESSIONActiveCounter:31...
7        Tomcat03             1120  2021-03-09 11:00:00 CST  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...
8            IG02              854  2021-03-09 11:00:00 CST  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:31; O...
9            IG01              837  2021-03-09 11:00:00 CST  JVM-Memory_7778_JVM_Memory_HeapMemoryMax:31; J...
10       apache02              823  2021-03-09 11:00:00 CST  OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT:3...
11       apache01              794  2021-03-09 11:00:00 CST  OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount:31;...
12           MG01              704  2021-03-09 11:00:00 CST  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:31; J...
13           MG02              699  2021-03-09 11:00:00 CST  JVM-Memory_7779_JVM_Memory_HeapMemoryMax:31; J...
14   ServiceTest6               65  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.cnt:5; app.mrt:4
15   ServiceTest3               64  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.mrt:4; app.cnt:4
16  ServiceTest10               64  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.mrt:4; app.cnt:4
17   ServiceTest1               64  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.mrt:4; app.cnt:4
18   ServiceTest2               64  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.mrt:4; app.cnt:4
19  ServiceTest11               64  2021-03-09 11:00:00 CST         app.sr:28; app.rr:28; app.mrt:4; app.cnt:4, 18704, (28268, 8))```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# reuse anomalies_window (from kernel). Ensure tz
tz = pytz.timezone('Asia/Shanghai')

# 1) Keep only anomaly rows within the window
anom = anomalies_window[anomalies_window['anomaly_flag']].copy()

# If no anomalies, create empty faults_table and aggregated summary
if anom.shape[0] == 0:
    faults_table = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes',
        'run_max_value','run_min_value','p95','p5','relative_deviation'
    ])
    faults_agg_top20 = pd.DataFrame(columns=['cmdb_id','number_of_faults','earliest_fault_start_dt','worst_relative_deviation','top_kpis'])
    total_faults = 0
else:
    # 1) Assign run indices per (cmdb_id, kpi_name) where consecutive timestamps differ by exactly 60 seconds
    anom_sorted = anom.sort_values(['cmdb_id', 'kpi_name', 'timestamp']).copy()
    def assign_run_idx(g):
        g = g.sort_values('timestamp').copy()
        # gap True when new run (first row or diff != 60)
        g['gap'] = g['timestamp'].diff().fillna(999).ne(60)
        # run_idx increments at each gap; create per-group run id
        g['run_idx'] = g['gap'].cumsum().astype(int)
        return g
    anom_runs_assigned = anom_sorted.groupby(['cmdb_id', 'kpi_name'], group_keys=False).apply(assign_run_idx)

    # 1) Aggregate each run
    agg = (
        anom_runs_assigned
        .groupby(['cmdb_id', 'kpi_name', 'run_idx'], as_index=False)
        .agg(
            run_start_dt=('timestamp_dt', 'min'),
            run_end_dt=('timestamp_dt', 'max'),
            run_length_minutes=('timestamp', 'count'),
            run_max_value=('value', 'max'),
            run_min_value=('value', 'min'),
            p95=('p95', 'first'),
            p5 = ('p5', 'first')
        )
    )

    # 2) Compute severity metric (relative_deviation)
    # Determine if run contains upper-tail and/or lower-tail anomalies by comparing run extremes to thresholds
    # Use tiny epsilon to avoid division by zero issues already handled by adding small denom in formula
    eps = 1e-9
    def compute_rel_dev(row):
        up_dev = np.nan
        low_dev = np.nan
        # upper-tail deviation if p95 available
        if pd.notna(row['p95']):
            up_dev = (row['run_max_value'] - row['p95']) / (abs(row['p95']) + eps)
        # lower-tail deviation if p5 available
        if pd.notna(row['p5']):
            low_dev = (row['p5'] - row['run_min_value']) / (abs(row['p5']) + eps)
        # Only consider positive deviations (breaches)
        up_dev_pos = up_dev if pd.notna(up_dev) and up_dev > 0 else -np.inf
        low_dev_pos = low_dev if pd.notna(low_dev) and low_dev > 0 else -np.inf
        best = max(up_dev_pos, low_dev_pos)
        return (np.nan if best == -np.inf else best)
    agg['relative_deviation'] = agg.apply(compute_rel_dev, axis=1)

    # Convert run_start_dt/run_end_dt to timezone-aware strings for storage/display (keep tz info)
    agg['run_start_ts'] = agg['run_start_dt'].dt.tz_convert(tz)
    agg['run_end_ts'] = agg['run_end_dt'].dt.tz_convert(tz)

    # 3) Filter runs: length >=2 minutes AND relative_deviation > 0.5
    faults_table = agg[
        (agg['run_length_minutes'] >= 2) &
        (agg['relative_deviation'] > 0.5)
    ].copy()

    # Keep requested columns and order
    faults_table = faults_table[[
        'cmdb_id','kpi_name','run_start_ts','run_end_ts','run_length_minutes',
        'run_max_value','run_min_value','p95','p5','relative_deviation'
    ]].reset_index(drop=True)

    # 4) faults_table stored in kernel variable (already assigned)

    # 5) Produce compact aggregate per component
    if faults_table.shape[0] == 0:
        faults_agg_top20 = pd.DataFrame(columns=['cmdb_id','number_of_faults','earliest_fault_start_dt','worst_relative_deviation','top_kpis'])
        total_faults = 0
    else:
        # number_of_faults and earliest start
        comp_agg = faults_table.groupby('cmdb_id').agg(
            number_of_faults=('cmdb_id','count'),
            earliest_fault_start_dt=('run_start_ts','min'),
            worst_relative_deviation=('relative_deviation','max')
        ).reset_index()

        # top 5 kpis by number_of_faults per component
        kpi_counts = faults_table.groupby(['cmdb_id','kpi_name']).size().reset_index(name='kpi_fault_count')

        def top_kpis_string(df, topn=5):
            df_sorted = df.sort_values('kpi_fault_count', ascending=False).head(topn)
            return "; ".join([f"{r['kpi_name']}:{int(r['kpi_fault_count'])}" for _, r in df_sorted.iterrows()])

        top_kpis_per_comp = (
            kpi_counts
            .groupby('cmdb_id')
            .apply(lambda d: pd.Series({'top_kpis': top_kpis_string(d, topn=5)}))
            .reset_index()
        )

        faults_agg = comp_agg.merge(top_kpis_per_comp, on='cmdb_id', how='left')

        # Format earliest timestamp as readable string in Asia/Shanghai timezone
        faults_agg['earliest_fault_start_dt'] = faults_agg['earliest_fault_start_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

        # Select and sort by number_of_faults desc, limit top 20
        faults_agg_top20 = faults_agg[['cmdb_id','number_of_faults','earliest_fault_start_dt','worst_relative_deviation','top_kpis']].sort_values('number_of_faults', ascending=False).head(20).reset_index(drop=True)

        total_faults = int(faults_table.shape[0])

# Final compact outputs: faults_agg_top20 and total_faults, and keep faults_table stored in kernel
faults_agg_top20, total_faults, faults_table.shape

```
Out[3]:
```
```python
# Plain-English summary of the fault-run detection results (using stored kernel variables)
lines = []
lines.append("Fault-run detection summary (incident window: 2021-03-09 11:00–11:30 UTC+8):")
lines.append(f"- Total fault runs meeting criteria (length>=2 min & relative_deviation>0.5): {total_faults}")
lines.append("- Top components by number of fault runs (component — count; earliest start; worst relative_deviation):")

# Use faults_agg_top20 from kernel, list its rows compactly
for i, r in faults_agg_top20.head(20).iterrows():
    comp = r['cmdb_id']
    cnt = int(r['number_of_faults'])
    start = r['earliest_fault_start_dt']
    worst = r['worst_relative_deviation']
    # format worst deviation compactly
    worst_s = f"{worst:.6g}" if pd.notna(worst) else "NaN"
    lines.append(f"  {i+1}) {comp} — {cnt} fault(s); earliest start: {start}; worst_relative_deviation: {worst_s}")

lines.append("- The full filtered fault runs are stored in variable 'faults_table'.")
lines.append("- Use 'faults_table' to inspect run_start/run_end, affected KPI, run length, values, and p5/p95 thresholds.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id  number_of_faults  earliest_fault_start_dt  worst_relative_deviation                                           top_kpis
0       Mysql02                 6  2021-03-09 11:25:00 CST              1.900000e+01  Mysql-MySQL_3306_Handler Read Next:1; Mysql-My...
1      Tomcat01                 2  2021-03-09 11:06:00 CST              6.000000e+10  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT...
2  ServiceTest8                 2  2021-03-09 11:21:00 CST              8.695396e-01                               app.cnt:1; app.mrt:1
3          MG02                 1  2021-03-09 11:09:00 CST              1.000000e+09   OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT:1
4          IG02                 1  2021-03-09 11:20:00 CST              1.481846e+00                           OSLinux-CPU_CPU_CPUWio:1
5       Redis02                 1  2021-03-09 11:02:00 CST              1.193082e+00                           OSLinux-CPU_CPU_CPUWio:1
6  ServiceTest9                 1  2021-03-09 11:19:00 CST              5.250000e-01                                          app.cnt:1, 14, (14, 10))```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

# Reuse timezone
tz = pytz.timezone('Asia/Shanghai')

# 1) Load trace.csv and compute global statistics grouped by (cmdb_id, trace_name)
trace_df = pd.read_csv('trace.csv')
trace_df['timestamp_dt'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

trace_grouped_stats_full = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(
        count='count',
        p50=lambda x: x.quantile(0.50),
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
    )
    .reset_index()
)

# Round numeric stats for compactness
num_cols = ['p50','p90','p95','p99']
trace_grouped_stats_full[num_cols] = trace_grouped_stats_full[num_cols].round(6)

# 2) Filter trace.csv to the incident window (UTC+8)
window_start = pd.Timestamp('2021-03-09 11:00:00', tz=tz)
window_end   = pd.Timestamp('2021-03-09 11:30:00', tz=tz)

trace_window = trace_df[
    (trace_df['timestamp_dt'] >= window_start) &
    (trace_df['timestamp_dt'] <= window_end)
].copy()

# 3) Join p95 from global grouped stats and flag anomalies where value >= p95
trace_window = trace_window.merge(
    trace_grouped_stats_full[['cmdb_id', 'trace_name', 'p95']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

trace_window['anomaly_flag'] = (
    (~trace_window['value'].isna()) &
    (~trace_window['p95'].isna()) &
    (trace_window['value'] >= trace_window['p95'])
)

# 4) Store full trace anomaly records for the window in 'trace_anomalies_window' with requested columns
trace_anomalies_window = trace_window.loc[:, ['timestamp', 'timestamp_dt', 'cmdb_id', 'trace_name', 'value', 'p95', 'anomaly_flag']].copy()

# 5) Produce compact aggregate of trace anomalies by cmdb_id
trace_anom_only = trace_anomalies_window[trace_anomalies_window['anomaly_flag']].copy()

if trace_anom_only.shape[0] == 0:
    trace_anom_agg_top20 = pd.DataFrame(columns=['cmdb_id','total_trace_anomalies','earliest_trace_anom_dt','top_traces'])
    total_trace_anomalies = 0
else:
    # total anomalies and earliest anomaly timestamp per component
    comp_agg = trace_anom_only.groupby('cmdb_id').agg(
        total_trace_anomalies=('anomaly_flag','sum'),
        earliest_trace_anom_dt=('timestamp_dt','min')
    ).reset_index()

    # top 5 trace_name contributors per component
    trace_counts = trace_anom_only.groupby(['cmdb_id','trace_name']).size().reset_index(name='trace_anom_count')

    def top_traces_string(df, topn=5):
        df_sorted = df.sort_values('trace_anom_count', ascending=False).head(topn)
        return "; ".join([f"{r['trace_name']}:{int(r['trace_anom_count'])}" for _, r in df_sorted.iterrows()])

    top_traces_per_comp = (
        trace_counts
        .groupby('cmdb_id')
        .apply(lambda d: pd.Series({'top_traces': top_traces_string(d, topn=5)}))
        .reset_index()
    )

    trace_anom_agg = comp_agg.merge(top_traces_per_comp, on='cmdb_id', how='left')
    trace_anom_agg['earliest_trace_anom_dt'] = trace_anom_agg['earliest_trace_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

    trace_anom_agg_top20 = trace_anom_agg.sort_values('total_trace_anomalies', ascending=False).head(20).reset_index(drop=True)
    total_trace_anomalies = int(trace_anom_only.shape[0])

# Final compact outputs: aggregated top20 and total count
trace_anom_agg_top20, total_trace_anomalies

```
Out[4]:
```
```python
# Compose a concise plain-English summary of the trace anomaly aggregate (using stored kernel variables)
lines = []
lines.append("Trace anomaly summary for incident window (2021-03-09 11:00–11:30 UTC+8):")
lines.append(f"- Total trace-anomaly records detected in the window: {int(total_trace_anomalies)}.")
lines.append("- Top components by trace anomaly count (component — count; earliest anomaly; top trace contributors):")

# Limit to top 10 for brevity
for i, r in trace_anom_agg_top20.head(10).iterrows():
    comp = r['cmdb_id']
    cnt = int(r['total_trace_anomalies'])
    earliest = r['earliest_trace_anom_dt']
    top_traces = r['top_traces'] if pd.notna(r['top_traces']) else ''
    lines.append(f"  {i+1}) {comp} — {cnt} anomalies; earliest: {earliest}; top traces: {top_traces}")

lines.append("- Full per-record anomalies are stored in 'trace_anomalies_window'. Global trace stats are in 'trace_grouped_stats_full'.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  total_trace_anomalies   earliest_trace_anom_dt                                         top_traces
0       MG01                     81  2021-03-09 11:00:00 CST  trace.from_Tomcat01.duration_mean:2; trace.fro...
1       MG02                     79  2021-03-09 11:00:00 CST  trace.from_dockerA1.row_count:3; trace.from_To...
2   dockerB2                     41  2021-03-09 11:00:00 CST  trace.self.duration_p95:10; trace.from_MG01.du...
3   dockerA1                     38  2021-03-09 11:00:00 CST  trace.self.duration_p95:5; trace.to_MG02.row_c...
4   Tomcat01                     34  2021-03-09 11:00:00 CST  trace.from_IG01.row_count:3; trace.from_IG01.d...
5   dockerB1                     34  2021-03-09 11:00:00 CST  trace.self.duration_p95:3; trace.from_MG01.dur...
6   Tomcat04                     33  2021-03-09 11:00:00 CST  trace.from_IG01.duration_mean:2; trace.from_IG...
7       IG02                     33  2021-03-09 11:00:00 CST  trace.self.duration_p95:2; trace.self.duration...
8   Tomcat02                     33  2021-03-09 11:00:00 CST  trace.from_IG01.duration_mean:2; trace.from_IG...
9   Tomcat03                     33  2021-03-09 11:00:00 CST  trace.from_IG01.duration_mean:2; trace.from_IG...
10  dockerA2                     33  2021-03-09 11:00:00 CST  trace.from_MG01.duration_mean:2; trace.from_MG...
11      IG01                     31  2021-03-09 11:00:00 CST  trace.to_Tomcat01.row_count:3; trace.self.dura..., 503)```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# Reuse existing variable: trace_anomalies_window (must exist in kernel)
tz = pytz.timezone('Asia/Shanghai')

# Components of interest
components = ['Mysql02', 'Tomcat01']

# Filter trace_anomalies_window for these components and only anomaly records
trace_anom_candidates = trace_anomalies_window[
    (trace_anomalies_window['cmdb_id'].isin(components)) &
    (trace_anomalies_window['anomaly_flag'] == True)
].copy()

# Ensure timestamp_dt is timezone-aware in Asia/Shanghai
if trace_anom_candidates.shape[0] > 0:
    trace_anom_candidates['timestamp_dt'] = pd.to_datetime(trace_anom_candidates['timestamp_dt']).dt.tz_convert(tz)

# 1) Per-component compact summary: total_trace_anomalies, earliest_trace_anom_dt, top 5 trace_name contributors
if trace_anom_candidates.shape[0] == 0:
    comp_summary = pd.DataFrame(columns=['cmdb_id','total_trace_anomalies','earliest_trace_anom_dt','top_traces'])
else:
    comp_agg = trace_anom_candidates.groupby('cmdb_id').agg(
        total_trace_anomalies=('anomaly_flag','sum'),
        earliest_trace_anom_dt=('timestamp_dt','min')
    ).reset_index()

    trace_counts = trace_anom_candidates.groupby(['cmdb_id','trace_name']).size().reset_index(name='trace_anom_count')

    def top_traces_string(df, topn=5):
        df_sorted = df.sort_values('trace_anom_count', ascending=False).head(topn)
        return "; ".join([f"{r['trace_name']}:{int(r['trace_anom_count'])}" for _, r in df_sorted.iterrows()])

    top_traces_per_comp = (
        trace_counts
        .groupby('cmdb_id')
        .apply(lambda d: pd.Series({'top_traces': top_traces_string(d, topn=5)}))
        .reset_index()
    )

    comp_summary = comp_agg.merge(top_traces_per_comp, on='cmdb_id', how='left')
    comp_summary['earliest_trace_anom_dt'] = comp_summary['earliest_trace_anom_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')

# Keep summary limited to the two requested components and in the original order
comp_summary = pd.DataFrame(components, columns=['cmdb_id']).merge(comp_summary, on='cmdb_id', how='left')

# 2) Up to 10 anomalous trace records per component (selected columns), sorted by timestamp ascending
sample_records_list = []
for c in components:
    subset = trace_anom_candidates[trace_anom_candidates['cmdb_id'] == c].sort_values('timestamp')
    sample = subset[['timestamp','timestamp_dt','cmdb_id','trace_name','value','p95']].head(10)
    sample_records_list.append(sample)

# Concatenate samples into one DataFrame with component separation (will be small)
sample_records = pd.concat(sample_records_list, ignore_index=True) if sample_records_list else pd.DataFrame(columns=['timestamp','timestamp_dt','cmdb_id','trace_name','value','p95'])

# Final compact outputs: comp_summary (2 rows) and sample_records (up to 20 rows, 10 per component)
comp_summary, sample_records

```
Out[5]:
```
```python
# Compose a concise plain-English summary using stored kernel variables (comp_summary, sample_records)
import pandas as pd

# Access comp_summary and sample_records from kernel
# Build summary lines
lines = []
lines.append("Trace-anomaly candidates summary for components ['Mysql02', 'Tomcat01'] (window 2021-03-09 11:00–11:30 UTC+8):")

# Mysql02 row
row_mysql = comp_summary[comp_summary['cmdb_id'] == 'Mysql02']
if row_mysql.empty or pd.isna(row_mysql.iloc[0]['total_trace_anomalies']):
    lines.append("- Mysql02: No trace-anomalies detected in the window.")
else:
    cnt = int(row_mysql.iloc[0]['total_trace_anomalies'])
    earliest = row_mysql.iloc[0]['earliest_trace_anom_dt']
    top = row_mysql.iloc[0]['top_traces'] if pd.notna(row_mysql.iloc[0]['top_traces']) else ""
    lines.append(f"- Mysql02: {cnt} anomalies; earliest: {earliest}; top traces: {top}")

# Tomcat01 row
row_tomcat = comp_summary[comp_summary['cmdb_id'] == 'Tomcat01']
if row_tomcat.empty or pd.isna(row_tomcat.iloc[0]['total_trace_anomalies']):
    lines.append("- Tomcat01: No trace-anomalies detected in the window.")
else:
    cnt = int(row_tomcat.iloc[0]['total_trace_anomalies'])
    earliest = row_tomcat.iloc[0]['earliest_trace_anom_dt']
    top = row_tomcat.iloc[0]['top_traces'] if pd.notna(row_tomcat.iloc[0]['top_traces']) else ""
    lines.append(f"- Tomcat01: {cnt} anomalies; earliest: {earliest}; top traces: {top}")

# Provide info about sample records returned
if sample_records.shape[0] == 0:
    lines.append("- No sample anomalous trace records returned for these components.")
else:
    # Count per component in the sample set
    counts = sample_records['cmdb_id'].value_counts().to_dict()
    for comp in ['Mysql02', 'Tomcat01']:
        c = counts.get(comp, 0)
        lines.append(f"- Sample records returned for {comp}: {c} row(s) (first anomalies by timestamp).")
    # Show the very first sample row (compact) if exists
    first_row = sample_records.sort_values('timestamp').iloc[0]
    fr_ts = pd.to_datetime(first_row['timestamp_dt']).tz_convert('Asia/Shanghai').strftime('%Y-%m-%d %H:%M:%S %Z')
    lines.append(f"- First sample anomaly: {first_row['cmdb_id']} | {fr_ts} | {first_row['trace_name']} | value={first_row['value']} p95={first_row['p95']}")

lines.append("- The full filtered trace-anomaly table for these two components is stored in 'trace_anom_candidates'.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  total_trace_anomalies   earliest_trace_anom_dt                                         top_traces
0   Mysql02                    NaN                      NaN                                                NaN
1  Tomcat01                   34.0  2021-03-09 11:00:00 CST  trace.from_IG01.row_count:3; trace.from_IG01.d...,     timestamp              timestamp_dt   cmdb_id                     trace_name        value          p95
0  1615258800 2021-03-09 11:00:00+08:00  Tomcat01      trace.from_IG02.row_count   311.000000   308.500000
1  1615258860 2021-03-09 11:01:00+08:00  Tomcat01      trace.from_IG01.row_count   308.000000   295.000000
2  1615258860 2021-03-09 11:01:00+08:00  Tomcat01  trace.from_root.duration_mean     0.075067     0.071314
3  1615258860 2021-03-09 11:01:00+08:00  Tomcat01   trace.from_root.duration_p95     0.378000     0.359100
4  1615258860 2021-03-09 11:01:00+08:00  Tomcat01      trace.from_root.row_count    15.000000    14.400000
5  1615258860 2021-03-09 11:01:00+08:00  Tomcat01           trace.self.row_count  4099.000000  3962.250000
6  1615258860 2021-03-09 11:01:00+08:00  Tomcat01        trace.to_MG02.row_count   308.000000   290.250000
7  1615259100 2021-03-09 11:05:00+08:00  Tomcat01      trace.from_IG01.row_count   295.000000   295.000000
8  1615259100 2021-03-09 11:05:00+08:00  Tomcat01        trace.to_MG02.row_count   291.000000   290.250000
9  1615259160 2021-03-09 11:06:00+08:00  Tomcat01   trace.from_IG01.duration_p95     3.665000     3.543050)```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# Reuse timezone
tz = pytz.timezone('Asia/Shanghai')

# Define incident window in UTC+8
window_start = pd.Timestamp('2021-03-09 11:00:00', tz=tz)
window_end   = pd.Timestamp('2021-03-09 11:30:00', tz=tz)

# 1) Load log.csv and filter for Tomcat01 in the window
log_df = pd.read_csv('log.csv')
log_df['timestamp_dt'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

logs_tomcat01_window = log_df[
    (log_df['cmdb_id'] == 'Tomcat01') &
    (log_df['timestamp_dt'] >= window_start) &
    (log_df['timestamp_dt'] <= window_end)
].copy()

# Aggregate per log_name: count and earliest timestamp in window
if logs_tomcat01_window.shape[0] > 0:
    logs_agg = (
        logs_tomcat01_window
        .groupby('log_name', as_index=False)
        .agg(count=('log_name','count'),
             earliest_log_ts_dt=('timestamp_dt','min'))
    )
    logs_agg['earliest_log_ts'] = logs_agg['earliest_log_ts_dt'].dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    logs_agg = logs_agg[['log_name','count','earliest_log_ts']].sort_values('count', ascending=False).reset_index(drop=True)
    logs_agg_top20 = logs_agg.head(20)
else:
    logs_agg_top20 = pd.DataFrame(columns=['log_name','count','earliest_log_ts'])

# Up to 20 sample log rows sorted by timestamp ascending (selected columns)
logs_samples = logs_tomcat01_window.sort_values('timestamp').loc[:, ['timestamp','timestamp_dt','cmdb_id','log_name','value']].head(20).reset_index(drop=True)

# 2) Load error_logs.csv and filter for Tomcat01 in the window
err_df = pd.read_csv('error_logs.csv')
err_df['timestamp_dt'] = pd.to_datetime(err_df['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

error_logs_tomcat01_window = err_df[
    (err_df['cmdb_id'] == 'Tomcat01') &
    (err_df['timestamp_dt'] >= window_start) &
    (err_df['timestamp_dt'] <= window_end)
].copy()

# Total number of error log rows and up to 20 sample error messages sorted by timestamp ascending
error_count = int(error_logs_tomcat01_window.shape[0])
error_samples = error_logs_tomcat01_window.sort_values('timestamp').loc[:, ['timestamp','timestamp_dt','cmdb_id','message']].head(20).reset_index(drop=True)

# Final compact outputs: logs aggregate, log samples, error count, error samples
logs_agg_top20, logs_samples, error_count, error_samples

```
Out[6]:
```
```python
# Concise plain-English summary of log filtering results for Tomcat01 (2021-03-09 11:00–11:30 UTC+8)
summary = (
    "Tomcat01 logs (2021-03-09 11:00–11:30 UTC+8):\n"
    "- Aggregated log counts by log_name (top):\n"
    "  * log.error_count: 27 rows, earliest 2021-03-09 11:00:00\n"
    "  * log.row_count:   27 rows, earliest 2021-03-09 11:00:00\n"
    "  * log.gc_pause_max:22 rows, earliest 2021-03-09 11:00:00\n"
    "- Sample log observations (first entries): log.row_count values around 500–600; log.error_count is 0 in the samples; gc_pause_max small (0.03–0.09s).\n"
    "- Error logs (error_logs.csv) for Tomcat01 in the window: 0 rows found.\n"
    "- Full filtered tables stored in kernel variables:\n"
    "  * logs_tomcat01_window  -- filtered log.csv rows for Tomcat01 in the window\n"
    "  * error_logs_tomcat01_window -- filtered error_logs.csv rows (empty)\n"
    "Use those variables to inspect full records if needed."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(           log_name  count          earliest_log_ts
0   log.error_count     27  2021-03-09 11:00:00 CST
1     log.row_count     27  2021-03-09 11:00:00 CST
2  log.gc_pause_max     22  2021-03-09 11:00:00 CST,      timestamp              timestamp_dt   cmdb_id          log_name   value
0   1615258800 2021-03-09 11:00:00+08:00  Tomcat01   log.error_count    0.00
1   1615258800 2021-03-09 11:00:00+08:00  Tomcat01  log.gc_pause_max    0.05
2   1615258800 2021-03-09 11:00:00+08:00  Tomcat01     log.row_count  569.00
3   1615258860 2021-03-09 11:01:00+08:00  Tomcat01   log.error_count    0.00
4   1615258860 2021-03-09 11:01:00+08:00  Tomcat01  log.gc_pause_max    0.06
5   1615258860 2021-03-09 11:01:00+08:00  Tomcat01     log.row_count  591.00
6   1615258920 2021-03-09 11:02:00+08:00  Tomcat01   log.error_count    0.00
7   1615258920 2021-03-09 11:02:00+08:00  Tomcat01  log.gc_pause_max    0.03
8   1615258920 2021-03-09 11:02:00+08:00  Tomcat01     log.row_count  554.00
9   1615258980 2021-03-09 11:03:00+08:00  Tomcat01   log.error_count    0.00
10  1615258980 2021-03-09 11:03:00+08:00  Tomcat01  log.gc_pause_max    0.06
11  1615258980 2021-03-09 11:03:00+08:00  Tomcat01     log.row_count  512.00
12  1615259040 2021-03-09 11:04:00+08:00  Tomcat01   log.error_count    0.00
13  1615259040 2021-03-09 11:04:00+08:00  Tomcat01  log.gc_pause_max    0.04
14  1615259040 2021-03-09 11:04:00+08:00  Tomcat01     log.row_count  507.00
15  1615259100 2021-03-09 11:05:00+08:00  Tomcat01   log.error_count    0.00
16  1615259100 2021-03-09 11:05:00+08:00  Tomcat01  log.gc_pause_max    0.05
17  1615259100 2021-03-09 11:05:00+08:00  Tomcat01     log.row_count  556.00
18  1615259160 2021-03-09 11:06:00+08:00  Tomcat01   log.error_count    0.00
19  1615259160 2021-03-09 11:06:00+08:00  Tomcat01  log.gc_pause_max    0.09, 0, Empty DataFrame
Columns: [timestamp, timestamp_dt, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse existing 'faults_table' from kernel
tz = pytz.timezone('Asia/Shanghai')

# Extract faults for Tomcat01 and store in kernel variable
faults_tomcat01 = faults_table[faults_table['cmdb_id'] == 'Tomcat01'].copy()

# Ensure run_start_ts/run_end_ts are timezone-aware and converted to Asia/Shanghai
if not faults_tomcat01.empty:
    # parse to datetime
    faults_tomcat01['run_start_ts'] = pd.to_datetime(faults_tomcat01['run_start_ts'])
    faults_tomcat01['run_end_ts'] = pd.to_datetime(faults_tomcat01['run_end_ts'])
    # localize if tz-naive, else convert
    if faults_tomcat01['run_start_ts'].dt.tz is None:
        faults_tomcat01['run_start_ts'] = faults_tomcat01['run_start_ts'].dt.tz_localize(tz)
    else:
        faults_tomcat01['run_start_ts'] = faults_tomcat01['run_start_ts'].dt.tz_convert(tz)
    if faults_tomcat01['run_end_ts'].dt.tz is None:
        faults_tomcat01['run_end_ts'] = faults_tomcat01['run_end_ts'].dt.tz_localize(tz)
    else:
        faults_tomcat01['run_end_ts'] = faults_tomcat01['run_end_ts'].dt.tz_convert(tz)

# Prepare compact table (limit 20 rows) sorted by run_start_ts ascending
cols_out = ['cmdb_id', 'kpi_name', 'run_start_ts', 'run_end_ts', 'run_length_minutes', 'run_max_value', 'p95', 'relative_deviation']
faults_tomcat01_compact = faults_tomcat01.loc[:, cols_out].sort_values('run_start_ts', ascending=True).head(20).reset_index(drop=True)

# Summary: unique kpi_name values and count of fault runs per kpi_name
if faults_tomcat01.empty:
    kpi_summary = pd.DataFrame(columns=['kpi_name', 'fault_run_count'])
else:
    kpi_summary = faults_tomcat01.groupby('kpi_name').size().reset_index()
    kpi_summary.columns = ['kpi_name', 'fault_run_count']
    kpi_summary = kpi_summary.sort_values('fault_run_count', ascending=False).reset_index(drop=True)

# Display compact outputs
faults_tomcat01_compact, kpi_summary.head(50)

```
Out[7]:
```
```python
# Plain-English summary of Tomcat01 fault runs (based on stored kernel variables)
total_runs = int(faults_tomcat01.shape[0])

lines = []
lines.append(f"Tomcat01 fault-run summary (incident window):")
lines.append(f"- Total fault runs detected for Tomcat01: {total_runs}")

if total_runs > 0:
    lines.append("- Fault run details (kpi_name — start -> end; length_min; run_max; p95; relative_deviation):")
    for _, r in faults_tomcat01_compact.iterrows():
        kpi = r['kpi_name']
        start = pd.to_datetime(r['run_start_ts']).tz_convert('Asia/Shanghai').strftime('%Y-%m-%d %H:%M:%S %Z')
        end = pd.to_datetime(r['run_end_ts']).tz_convert('Asia/Shanghai').strftime('%Y-%m-%d %H:%M:%S %Z')
        length = int(r['run_length_minutes'])
        run_max = r['run_max_value']
        p95 = r['p95']
        rel = r['relative_deviation']
        lines.append(f"  - {kpi} — {start} -> {end}; {length} min; run_max={run_max:.6g}; p95={p95:.6g}; relative_deviation={rel:.6g}")

    # Unique KPI list and counts
    lines.append("- Fault runs count per KPI:")
    for _, kr in kpi_summary.iterrows():
        lines.append(f"  - {kr['kpi_name']}: {int(kr['fault_run_count'])} run(s)")
else:
    lines.append("- No fault runs found for Tomcat01.")

lines.append("- The full extracted runs table is stored in variable 'faults_tomcat01' for further inspection.")
summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id                                        kpi_name              run_start_ts                run_end_ts  run_length_minutes  run_max_value  p95  relative_deviation
0  Tomcat01  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT 2021-03-09 11:06:00+08:00 2021-03-09 11:08:00+08:00                   3            1.0  0.0        1.000000e+09
1  Tomcat01    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT 2021-03-09 11:20:00+08:00 2021-03-09 11:21:00+08:00                   2           60.0  0.0        6.000000e+10,                                          kpi_name  fault_run_count
0  OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                1
1    OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                1)```
```